# 12 - Hypothesis Results

Name: Usha Oudeacoumar
Student ID: 20057009
Email: 20057009@mydbs.ie

This notebook does not run any new models. It just reads the result tables that the
other notebooks already saved in outputs/tables, and checks them against the 4
hypotheses from the project spec (H1-H4). The idea is to have one place to look at
when I just want the answer to "did we prove H1/H2/H3/H4 or not", instead of digging
through notebooks 06-10 every time. This notebook should be re-run (or at least
re-read) after notebooks 08, 09 and 10 are done, since right now only H1 can be
answered (the rest are marked as pending below).

In [1]:
import sys
sys.path.append(".")
import os
import pandas as pd
from utils import OUT_TABLES, OUT_REPORTS

os.makedirs(OUT_REPORTS, exist_ok=True)

internal_val = pd.read_csv(os.path.join(OUT_TABLES, "06_internal_validation_metrics.csv"))
drift = pd.read_csv(os.path.join(OUT_TABLES, "07_temporal_drift_metrics.csv"))
drift

,yr,model,n,auroc,brier,cal_intercept,cal_slope,ece
0,2015,logistic_regression,16393,0.813589,0.080459,-0.284415,0.984055,0.021988
1,2015,gradient_boosting,16393,0.821926,0.078495,-0.180894,1.044735,0.020618
2,2016,logistic_regression,15242,0.809096,0.078984,-0.323017,0.992952,0.026389
3,2016,gradient_boosting,15242,0.816619,0.077736,-0.229849,1.047231,0.025151
4,2017,logistic_regression,12793,0.797547,0.089752,-0.213976,0.945089,0.012770
5,2017,gradient_boosting,12793,0.807454,0.087589,-0.087161,1.022471,0.011168
6,2018,logistic_regression,15827,0.792851,0.099716,-0.091435,0.955404,0.008709
7,2018,gradient_boosting,15827,0.799184,0.098327,-0.003746,1.004207,0.003225
8,2019,logistic_regression,14899,0.797870,0.093942,-0.182984,0.979252,0.014852
9,2019,gradient_boosting,14899,0.805235,0.092004,-0.081070,1.040731,0.013348


## H1 - Temporal calibration drift

Claim: a model trained on historical data (2011-2014) will show measurable calibration
deterioration on later years (2015-2019), if we freeze it and don't touch it.

Check: compare the internal validation metrics (notebook 06, same distribution the
model was trained on) against the frozen-model metrics on each eval year
(notebook 07). If calibration on eval years is clearly worse than internal
validation, H1 is supported.

In [2]:
# internal validation is basically "year 0" baseline, put it side by side with
# the eval years so the drift is obvious
internal_val_marked = internal_val.copy()
internal_val_marked["yr"] = "internal_val"

h1_compare = pd.concat([internal_val_marked, drift], ignore_index=True)
h1_compare = h1_compare[["yr", "model", "auroc", "brier", "cal_intercept", "cal_slope", "ece"]]
h1_compare

,yr,model,auroc,brier,cal_intercept,cal_slope,ece
0,internal_val,logistic_regression,0.820875,0.092033,0.017647,1.006712,0.009267
1,internal_val,gradient_boosting,0.829765,0.090150,0.105577,1.066421,0.009966
2,2015,logistic_regression,0.813589,0.080459,-0.284415,0.984055,0.021988
3,2015,gradient_boosting,0.821926,0.078495,-0.180894,1.044735,0.020618
4,2016,logistic_regression,0.809096,0.078984,-0.323017,0.992952,0.026389
5,2016,gradient_boosting,0.816619,0.077736,-0.229849,1.047231,0.025151
6,2017,logistic_regression,0.797547,0.089752,-0.213976,0.945089,0.012770
7,2017,gradient_boosting,0.807454,0.087589,-0.087161,1.022471,0.011168
8,2018,logistic_regression,0.792851,0.099716,-0.091435,0.955404,0.008709
9,2018,gradient_boosting,0.799184,0.098327,-0.003746,1.004207,0.003225


**H1 verdict: SUPPORTED.**

- cal_intercept (ideal 0): internal validation was 0.018 (log reg) / 0.106 (gb),
  basically fine. On the eval years it jumps to between -0.32 and -0.004,
  mostly a lot further from 0 than internal validation, and negative, meaning
  the frozen model over-predicts admission risk once applied to later years.
- auroc gets worse every year from 2015 to 2018 for both models (log reg
  0.814 -> 0.793, gb 0.822 -> 0.799), then recovers slightly in 2019.
- brier gets worse every year from 2015 to 2018 too (log reg 0.080 -> 0.100,
  gb 0.078 -> 0.098), matching the auroc pattern, then improves a bit in 2019.
- ece does not move the same way as brier/auroc, 2018 is actually the best
  ece year for both models even though it's the worst brier/auroc year. This
  is a good reminder that calibration metrics and discrimination metrics
  don't have to move together.

So yes, a model trained on 2011-2014 and left untouched clearly gets worse over
time on both calibration and discrimination, confirming H1.

## H2 - Recalibration effectiveness

Claim: retrospective recalibration (intercept-only, intercept+slope, Platt, isotonic)
will improve calibration compared to leaving the model unchanged.

**Status: PENDING.** Needs notebook 08 (apply_recalibration) to be built and run
first. Once that saves a results table (eg outputs/tables/08_recalibration_metrics.csv),
add a cell here comparing it against the "no recalibration" numbers already in
drift above.

## H3 - Recalibration versus retraining

Claim: some recalibration approaches may get close to the calibration performance
of full retraining, at a lower computational cost.

**Status: PENDING.** Needs notebook 09 (retrain_models, rolling 4 year retrain) in
addition to notebook 08. Compare recalibrated-frozen-model metrics vs retrained-model
metrics per eval year.

## H4 - Temporal adaptation

Claim: more recent calibration data may adapt better to changing distributions than
older calibration data, but this has to be tested and not assumed.

**Status: PENDING.** This is really Protocol A (fixed historical calibration set)
vs Protocol B (rolling prior-year window) from notebook 08, compared against each
other, not against the no-recalibration baseline.